# 音频特征工程与经典方法

## 学习目标

- 理解音频信号从原始波形到模型输入的完整转换流程
- 掌握MFCC、梅尔语谱图等关键音频特征
- 理解梅尔刻度与CI电极映射的关系
- 用torchaudio进行音频加载、变换和特征提取
- 对比不同特征在分类任务中的效果

## 1. 音频信号基础

音频信号是一维时间序列——振幅随时间变化。深度学习中通常先做特征提取，把一维波形转换为更适合模型处理的表示。

转换链：
```
原始波形 -> 预加重 -> 分帧加窗 -> FFT -> 梅尔滤波 -> 对数 -> (DCT -> MFCC)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
import torchaudio.transforms as T
import os

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 4)

ESC50_DIR = '../data/ESC-50'
if os.path.exists(os.path.join(ESC50_DIR, 'audio')):
    print('ESC-50数据集已就绪')
else:
    print('ESC-50未找到，部分示例将使用合成信号')
    ESC50_DIR = None

In [ ]:
# 加载一条真实音频
if ESC50_DIR:
    import pandas as pd
    df = pd.read_csv(os.path.join(ESC50_DIR, 'meta', 'esc50.csv'))
    row = df[df['category'] == 'dog'].iloc[0]
    audio_path = os.path.join(ESC50_DIR, 'audio', row['filename'])
    waveform, sr = torchaudio.load(audio_path)
    print('文件:', row['filename'], '类别:', row['category'])
else:
    sr = 44100
    t = np.linspace(0, 5, sr*5, endpoint=False)
    waveform = torch.FloatTensor(0.3*np.sin(2*np.pi*440*t)+0.1*np.random.randn(len(t))).unsqueeze(0)
    print('使用合成信号(440Hz + noise)')

print('采样率:', sr, 'Hz')
print('波形形状:', waveform.shape)
print('时长: %.2f 秒' % (waveform.shape[1]/sr))

plt.figure(figsize=(12, 3))
time_axis = np.arange(waveform.shape[1]) / sr
plt.plot(time_axis, waveform[0].numpy())
plt.xlabel('Time (s)'); plt.ylabel('Amplitude'); plt.title('Waveform')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 2. 从波形到语谱图的完整流程

逐步走完从原始波形到语谱图的每一步。

### 步骤总览
```
原始波形 -> 预加重 -> 分帧加窗 -> FFT -> 梅尔滤波 -> 对数 -> MFCC
```

In [ ]:
# 步骤1: 预加重 y[n] = x[n] - alpha * x[n-1], alpha=0.97
alpha = 0.97
waveform_np = waveform[0].numpy()
emphasized = np.append(waveform_np[0], waveform_np[1:] - alpha * waveform_np[:-1])

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].plot(time_axis, waveform_np, alpha=0.7)
axes[0].set_title('Original'); axes[0].set_xlabel('Time (s)'); axes[0].grid(True, alpha=0.3)
axes[1].plot(time_axis, emphasized, alpha=0.7, color='orange')
axes[1].set_title('Pre-emphasized (alpha=0.97)'); axes[1].set_xlabel('Time (s)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('高频细节变得更明显了')

In [ ]:
# 步骤2: 分帧 + 加窗 (帧长25ms, 帧移10ms, 汉宁窗)
frame_length = int(0.025 * sr)
frame_step = int(0.010 * sr)
n_frames = 1 + (len(emphasized) - frame_length) // frame_step
window = np.hanning(frame_length)

frames = []
for i in range(n_frames):
    start = i * frame_step
    frame = emphasized[start:start+frame_length] * window
    frames.append(frame)
frames = np.array(frames)

print('帧长:', frame_length, 'samples (%.1f ms)' % (frame_length/sr*1000))
print('帧移:', frame_step, 'samples (%.1f ms)' % (frame_step/sr*1000))
print('总帧数:', n_frames)
print('分帧后形状:', frames.shape)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for i in range(3):
    axes[i].plot(frames[i])
    axes[i].set_title('Frame %d' % i)
    axes[i].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 步骤3: FFT -> 功率谱
NFFT = 2048
mag_frames = np.absolute(np.fft.rfft(frames, NFFT))
pow_frames = (1.0 / NFFT) * (mag_frames ** 2)

print('功率谱形状:', pow_frames.shape)
print('频率分辨率: %.1f Hz' % (sr/NFFT))
print('最高频率: %.1f Hz' % (sr/2))

freq_axis = np.fft.rfftfreq(NFFT, 1/sr)
plt.figure(figsize=(10, 3))
plt.plot(freq_axis[:200], 10*np.log10(pow_frames[0][:200]+1e-10))
plt.xlabel('Frequency (Hz)'); plt.ylabel('Power (dB)'); plt.title('Power Spectrum of Frame 0')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 步骤4: 梅尔滤波器组
# 梅尔刻度模拟人耳感知: 低频高分辨率, 高频低分辨率
n_mels = 40
mel_transform_manual = T.MelSpectrogram(
    sample_rate=sr, n_fft=NFFT, hop_length=frame_step,
    n_mels=n_mels, f_min=0, f_max=sr//2
)

# 可视化梅尔滤波器组
fbank = mel_transform_manual.mel_scale.fb.numpy()
plt.figure(figsize=(10, 4))
for i in range(min(n_mels, 20)):
    plt.plot(freq_axis, fbank[i])
plt.xlabel('Frequency (Hz)'); plt.ylabel('Filter weight')
plt.title('Mel Filterbank (first 20 of %d)' % n_mels)
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print('低频区域滤波器密集(高分辨率), 高频区域稀疏(低分辨率)')

In [ ]:
# 步骤5: 应用梅尔滤波 -> 对数变换 = Log-Mel Spectrogram
mel_spec = mel_transform_manual(waveform)
db_transform = T.AmplitudeToDB(stype='power', top_db=80)
log_mel_spec = db_transform(mel_spec)

print('Mel spectrogram shape:', mel_spec.shape)

plt.figure(figsize=(12, 4))
plt.imshow(log_mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='Magnitude (dB)')
plt.xlabel('Time Frame'); plt.ylabel('Mel Band'); plt.title('Log-Mel Spectrogram')
plt.tight_layout(); plt.show()

In [ ]:
# 步骤6: DCT -> MFCC
mfcc_transform = T.MFCC(
    sample_rate=sr, n_mfcc=13,
    melkwargs={'n_fft': NFFT, 'hop_length': frame_step, 'n_mels': n_mels}
)
mfcc = mfcc_transform(waveform)
print('MFCC shape:', mfcc.shape)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].imshow(log_mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Log-Mel Spectrogram'); axes[0].set_ylabel('Mel Band')
axes[1].imshow(mfcc[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('MFCC (13 coefficients)'); axes[1].set_ylabel('MFCC Index')
axes[1].set_xlabel('Time Frame')
plt.tight_layout(); plt.show()

## 3. 梅尔刻度与CI电极映射

梅尔刻度与CI电极频率分配高度相关——这是用梅尔语谱图做CI研究的物理基础。

CI电极排列：底转(高频,窄间距) -> 顶转(低频,宽间距)
梅尔刻度：低频(高分辨率) -> 高频(低分辨率)

梅尔语谱图的每个频带，大致对应CI的一个电极通道。

In [ ]:
def hz_to_mel(hz): return 2595 * np.log10(1 + hz / 700)
def mel_to_hz(mel): return 700 * (10 ** (mel / 2595) - 1)

freqs_linear = np.linspace(0, 8000, 1000)
mel_values = hz_to_mel(freqs_linear)

# CI典型电极频率边界(22电极)
n_electrodes = 22
mel_bounds = np.linspace(hz_to_mel(250), hz_to_mel(8000), n_electrodes + 1)
electrode_freq_bounds = mel_to_hz(mel_bounds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(freqs_linear, mel_values, 'b-', linewidth=2)
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Mel')
axes[0].set_title('Mel Scale vs Linear Frequency')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=hz_to_mel(250), color='r', linestyle='--', alpha=0.5)
axes[0].axhline(y=hz_to_mel(8000), color='r', linestyle='--', alpha=0.5)

for i in range(n_electrodes):
    low = electrode_freq_bounds[i]
    high = electrode_freq_bounds[i+1]
    axes[1].barh(i, high-low, left=low, height=0.8, color='C%d' % (i%10), alpha=0.7)
axes[1].set_xlabel('Frequency Band Width (Hz)')
axes[1].set_ylabel('Electrode # (apex=0, base=21)')
axes[1].set_title('CI Electrode Freq Allocation (%d ch, Mel-spaced)' % n_electrodes)
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('低频电极分配窄频带(高分辨率), 高频电极分配宽频带(低分辨率)')
print('这和梅尔刻度完全一致! 梅尔语谱图天然适合CI研究。')

## 4. torchaudio实战

`torchaudio.transforms` 提供一站式特征提取工具。之后的模型代码全部使用torchaudio的变换。

In [ ]:
# torchaudio常用变换速查
print('=== torchaudio.transforms ===')
print('MelSpectrogram  : waveform -> mel spectrogram')
print('MFCC            : waveform -> MFCC')
print('Spectrogram     : waveform -> spectrogram')
print('AmplitudeToDB   : linear -> dB')
print('FrequencyMasking: freq mask (augmentation)')
print('TimeMasking     : time mask (augmentation)')
print('Resample        : resampling')
print()
print('=== torchaudio.functional ===')
print('amplitude_to_DB : amplitude to dB (function)')
print('compute_deltas  : delta features')

In [ ]:
# 实战: 提取三种特征并对比
if ESC50_DIR:
    row = df[df['category'] == 'crying_baby'].iloc[0]
    wav, sr_esc = torchaudio.load(os.path.join(ESC50_DIR, 'audio', row['filename']))
    wav_16k = T.Resample(sr_esc, 16000)(wav)
    sr_target = 16000
else:
    sr_target = 16000
    t = np.linspace(0, 2, 32000, endpoint=False)
    wav_16k = torch.FloatTensor(0.3*np.sin(2*np.pi*300*t)+0.15*np.sin(2*np.pi*900*t)+0.05*np.random.randn(len(t))).unsqueeze(0)

n_fft, hop_length, n_mels = 512, 160, 64

mel_t = T.MelSpectrogram(sr_target, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
spec_t = T.Spectrogram(n_fft=n_fft, hop_length=hop_length)
mfcc_t = T.MFCC(sr_target, n_mfcc=13, melkwargs={'n_fft':n_fft,'hop_length':hop_length,'n_mels':n_mels})
db_transform = T.AmplitudeToDB(stype='power', top_db=80)

mel_spec = db_transform(mel_t(wav_16k))
raw_spec = db_transform(spec_t(wav_16k))
mfcc_feat = mfcc_t(wav_16k)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].imshow(raw_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Linear Spectrogram'); axes[0].set_ylabel('Freq Bin')
axes[1].imshow(mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('Mel Spectrogram'); axes[1].set_ylabel('Mel Band')
axes[2].imshow(mfcc_feat[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[2].set_title('MFCC'); axes[2].set_ylabel('MFCC Index')
axes[2].set_xlabel('Time Frame')
plt.tight_layout(); plt.show()

print('Linear Spectrogram:', raw_spec.shape)
print('Mel Spectrogram:', mel_spec.shape)
print('MFCC:', mfcc_feat.shape)

### 三种特征对比

| 特征 | 维度 | 优点 | 缺点 | 适用场景 |
|------|------|------|------|----------|
| 线性语谱图 | [n_fft/2+1, T] | 保留完整频率信息 | 维度高 | 精细频率分析 |
| 梅尔语谱图 | [n_mels, T] | 符合人耳感知，维度低 | 丢失部分高频细节 | **深度学习首选** |
| MFCC | [n_mfcc, T] | 紧凑，去相关 | 丢失相位信息 | 传统语音识别 |

在深度学习时代，梅尔语谱图是绝对主流。

## 5. 特征对比实验

用同一个分类任务对比三种特征的效果。任务：区分ESC-50中两类声音。

In [ ]:
if not ESC50_DIR:
    print('需要ESC-50数据集才能运行此实验，请参见 DATA_REQUIREMENTS.md')
else:
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    cat1, cat2 = 'dog', 'rooster'
    df_sub = df[df['category'].isin([cat1, cat2])].reset_index(drop=True)
    print('类别: %s (0) vs %s (1)' % (cat1, cat2))
    print('样本数:', len(df_sub))

    class ESC50BinaryDataset(Dataset):
        def __init__(self, df, data_dir, feature_type='mel', n_mels=64, n_mfcc=13):
            self.df = df; self.data_dir = data_dir
            self.feature_type = feature_type
            self.n_mels = n_mels; self.n_mfcc = n_mfcc
            self.sr_target = 16000
            self.resampler_cache = {}
            self.db_transform = T.AmplitudeToDB(stype='power', top_db=80)
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            wav, orig_sr = torchaudio.load(os.path.join(self.data_dir, 'audio', row['filename']))
            if orig_sr != self.sr_target:
                if orig_sr not in self.resampler_cache:
                    self.resampler_cache[orig_sr] = T.Resample(orig_sr, self.sr_target)
                wav = self.resampler_cache[orig_sr](wav)
            if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
            if self.feature_type == 'mel':
                t = T.MelSpectrogram(self.sr_target, n_fft=512, hop_length=160, n_mels=self.n_mels)
                feat = self.db_transform(t(wav))
            elif self.feature_type == 'mfcc':
                t = T.MFCC(self.sr_target, n_mfcc=self.n_mfcc, melkwargs={'n_fft':512,'hop_length':160,'n_mels':self.n_mels})
                feat = t(wav)
            elif self.feature_type == 'wave':
                target_len = self.sr_target * 5
                if wav.shape[1] < target_len: wav = torch.nn.functional.pad(wav, (0, target_len-wav.shape[1]))
                else: wav = wav[:, :target_len]
                feat = wav
            label = 0 if row['category'] == cat1 else 1
            return feat, label
    print('数据集类定义完成')

In [ ]:
if ESC50_DIR:
    class SimpleCNN(nn.Module):
        def __init__(self, n_classes=2):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)),
            )
            self.classifier = nn.Linear(32, n_classes)
        def forward(self, x):
            if x.dim() == 3: x = x.unsqueeze(1)
            f = self.features(x).view(x.size(0), -1)
            return self.classifier(f)

    import torch.optim as optim
    results = {}
    for feat_type in ['mel', 'mfcc', 'wave']:
        print('\n--- %s ---' % feat_type)
        dataset = ESC50BinaryDataset(df_sub, ESC50_DIR, feature_type=feat_type)
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=16)
        model = SimpleCNN()
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        for epoch in range(15):
            model.train()
            for bx, by in train_loader:
                out = model(bx); loss = criterion(out, by)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for bx, by in val_loader:
                correct += (model(bx).argmax(1) == by).sum().item()
                total += by.size(0)
        acc = correct / total; results[feat_type] = acc
        print('%s accuracy: %.3f' % (feat_type, acc))
    print('\n=== 对比结果 ===')
    for k,v in results.items(): print('%8s: %.3f' % (k, v))
    print('通常: mel >= mfcc > wave')

## 6. 实操练习

任务：批量提取ESC-50所有音频的梅尔语谱图特征。

数据流：
```
WAV文件 -> torchaudio.load() -> Resample(16kHz) -> MelSpectrogram -> AmplitudeToDB -> 保存为.pt文件
```

**请自行完成代码**，课堂上只提供数据流图和API提示。

In [ ]:
# 实操练习: 批量提取ESC-50梅尔语谱图
# 提示:
#   1. 读取 esc50.csv 获取文件列表和标签
#   2. 用 torchaudio.load() 加载每个文件
#   3. 重采样到16kHz
#   4. 提取64维梅尔语谱图
#   5. 保存为 .pt 文件 (torch.save)
#
# === 请在下方编写代码 ===

print('请自行完成代码')
print('完成后, data/processed/ 目录下应有 esc50_mel.pt')
print('该文件应包含: features tensor, labels tensor, fold tensor')

## 本节要点

1. 音频特征提取链：波形 -> 预加重 -> 分帧加窗 -> FFT -> 梅尔滤波 -> 对数 -> MFCC
2. 梅尔语谱图是深度学习首选特征——符合人耳感知，维度适中
3. 梅尔刻度与CI电极映射高度相关——这是用梅尔语谱图做CI研究的物理基础
4. torchaudio.transforms 提供一站式特征提取
5. 不同特征对比：mel >= mfcc > wave

---
下一节：[02-crnn-classifier.ipynb](02-crnn-classifier.ipynb)